In [3]:
!pip install -q "google-adk[extensions]"

In [4]:
!pip install mcp

In [3]:
import os
import time
import warnings
from IPython.display import clear_output, display as ipy_display
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
import ipywidgets as widgets
import requests
import vertexai

# Suppress experimental JSON schema warning
warnings.filterwarnings(
    "ignore",
    message=".*JSON_SCHEMA_FOR_FUNC_DECL is enabled.*"
)

# ------------------------------------------------------------
# 1. INITIALIZATION & SETUP
# ------------------------------------------------------------

# Retrieve current GCP Project ID dynamically
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

# Fallback if unset
if PROJECT_ID == "(unset)" or not PROJECT_ID:
    PROJECT_ID = "qwiklabs-gcp-03-a6826a21a443"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
os.environ["ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS"] = "true"

vertexai.init(project=PROJECT_ID, location=LOCATION)

session_service = InMemorySessionService()

# ------------------------------------------------------------
# 2. DEFINE THE TWO AGENTS & RUNNERS (FLASH & PRO)
# ------------------------------------------------------------

agent_instruction = (
    "You are a helpful weather assistant. "
    "When a user provides a location, use your geocoding and forecast tools "
    "to find the weather forecast for that location. "
    "If the location is generic (like 'Chicago' or 'Lancaster'), proceed with "
    "geocoding the most probable option. "
    "IMPORTANT: Always explicitly state the identified state (or region) "
    "and country for the city you chose in your final response to the user, "
    "so they know exactly which location's forecast you retrieved (e.g., "
    "'Retrieving weather for Lancaster, Pennsylvania, United States'). "
    "If the user clarifies a specific state or country, use that precise "
    "location for geocoding."
)

# Flash Agent and Runner
agent_flash = LlmAgent(
    name="VertexFlashWeather",
    model="vertex_ai/gemini-2.5-flash",
    instruction=agent_instruction,
)

runner_flash = Runner(
    agent=agent_flash,
    app_name="VertexFlashApp",
    session_service=session_service,
    auto_create_session=True,
)

# Pro Agent and Runner
agent_pro = LlmAgent(
    name="VertexProWeather",
    model="vertex_ai/gemini-2.5-pro",
    instruction=agent_instruction,
)

runner_pro = Runner(
    agent=agent_pro,
    app_name="VertexProApp",
    session_service=session_service,
    auto_create_session=True,
)

# ------------------------------------------------------------
# 3. DEFINE ADK TOOLS AND ASSIGN TO BOTH AGENTS
# ------------------------------------------------------------


def geocode_place_free(address: str) -> str:
    """Geocode an address into latitude and longitude coordinates."""
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }

    try:
        response = requests.get(
            base_url, params=params, headers=headers, timeout=10
        )
        response.raise_for_status()
        data = response.json()
        if data:
            return f"{data[0]['lat']}, {data[0]['lon']}"
    except Exception as e:
        return f"Error geocoding location: {e}"

    return "Error: Location not found."


def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Retrieve the National Weather Service (NWS) forecast."""
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"

    try:
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        forecast_response = requests.get(
            forecast_url, headers=headers, timeout=10
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]

        formatted = []
        for period in periods[:2]:
            formatted.append(
                f"{period['name']}: {period['temperature']}°"
                f"{period['temperatureUnit']}, {period['shortForecast']}. "
                f"Detailed: {period['detailedForecast']}"
            )
        return "\n\n".join(formatted)

    except Exception as e:
        return f"Error retrieving forecast: {e}"


agent_flash.tools = [geocode_place_free, get_nws_forecast]
agent_pro.tools = [geocode_place_free, get_nws_forecast]

# ------------------------------------------------------------
# 4. FULLY INTERACTIVE WIDGET-BASED CHAT INTERFACE WITH STACKED LAYOUT
# ------------------------------------------------------------

session_id = "weather_session"
user_id = "russell"

# Chat elements setup
chat_history = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=[("Gemini 2.5 Flash", "flash"), ("Gemini 2.5 Pro", "pro")],
    value="flash",
    description="Select Model",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="260px"),
)

# Inject Custom High-Contrast Dark CSS style
custom_css = """
<style>
.widget-dropdown select {
    background-color: #212529 !important;
    color: #ffffff !important;
    border: 1px solid #495057 !important;
    border-radius: 4px !important;
}
.widget-dropdown select option {
    background-color: #212529 !important;
    color: #ffffff !important;
}
</style>
"""
css_widget = widgets.HTML(custom_css)

user_text_input = widgets.Text(
    placeholder="Type your location or reply here and press Enter...",
    layout=widgets.Layout(width="65%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="info",
    layout=widgets.Layout(width="15%"),
)

exit_button = widgets.Button(
    description="End Chat",
    button_style="danger",
    layout=widgets.Layout(width="15%"),
)


def append_message(sender, text, is_agent=False):
    """Appends formatted messages to the interactive chat history widget."""
    with chat_history:
        color = "#0d6efd" if not is_agent else "#198754"
        sender_name = "You" if not is_agent else "Agent"
        display_text = (
            f"<b style='color: {color};'>{sender_name}:</b> {text}<br>"
        )
        ipy_display(widgets.HTML(display_text))


def disable_interface():
    """Disable all widgets to formally conclude the session state."""
    model_dropdown.disabled = True
    user_text_input.disabled = True
    send_button.disabled = True
    exit_button.disabled = True
    with chat_history:
        end_text = (
            "<hr><b style='color: #dc3545;'>Conversation ended. "
            "Please run the cell again to start a new chat.</b>"
        )
        ipy_display(widgets.HTML(end_text))


def handle_submit(sender):
    """Handles user interaction submission on 'Send' or text enter key."""
    user_query = user_text_input.value.strip()
    if not user_query:
        return

    # Clear input field immediately
    user_text_input.value = ""

    # Display user input
    append_message(user_id, user_query, is_agent=False)

    if user_query.lower() in ["exit", "quit"]:
        disable_interface()
        return

    # Select the runner based on active model dropdown option
    selected_model_key = model_dropdown.value
    if selected_model_key == "pro":
        active_runner = runner_pro
        model_display_name = "Gemini 2.5 Pro"
    else:
        active_runner = runner_flash
        model_display_name = "Gemini 2.5 Flash"

    # Log active model in chat metadata
    with chat_history:
        routed_text = (
            f"<small style='color: #6c757d; font-style: italic;'>"
            f"[Routing message to {model_display_name}]</small>"
        )
        ipy_display(widgets.HTML(routed_text))

    # Get response from active ADK runner
    message_content = types.Content(
        parts=[types.Part.from_text(text=user_query)]
    )

    try:
        events = active_runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content,
        )
        for event in events:
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        append_message("agent", part.text, is_agent=True)
    except Exception as e:
        append_message("system", f"Error sending message: {e}", is_agent=True)


def handle_exit(button):
    """Invokes the end chat routine when 'End Chat' is pressed."""
    disable_interface()


# Register listeners
user_text_input.on_submit(handle_submit)
send_button.on_click(handle_submit)
exit_button.on_click(handle_exit)

# Organizing into two clean control rows
top_row = widgets.HBox([model_dropdown])
bottom_row = widgets.HBox(
    [user_text_input, send_button, exit_button],
    layout=widgets.Layout(grid_gap="10px"),
)

control_box = widgets.VBox(
    [top_row, bottom_row],
    layout=widgets.Layout(grid_gap="10px", margin="10px 0px"),
)
interface_box = widgets.VBox([css_widget, chat_history, control_box])

clear_output()
ipy_display(interface_box)

with chat_history:
    welcome_text = (
        "<b style='color: #6c757d;'>Weather Agent initialized. "
        "Choose your model and type a location below:</b><br>"
    )
    ipy_display(widgets.HTML(welcome_text))

In [1]:
import time
import ipywidgets as widgets
from IPython.display import display as ipy_display
from google.genai import types

# Define output area and button
test_button = widgets.Button(
    description="Run Multi-City Test",
    button_style="primary",
    layout=widgets.Layout(width="250px")
)
test_output = widgets.Output()

def run_suite_on_button_click(b):
    test_button.disabled = True
    test_output.clear_output()

    with test_output:
        print("=== Starting Programmatic Multi-City Weather Test Suite ===\n")
        cities = ['Chicago', 'Lancaster, PA', 'Miami']
        # Modified to only include Gemini 2.5 Flash
        runners = [
            ("Gemini 2.5 Flash", runner_flash)
        ]

        for model_name, runner in runners:
            print(f"-> Initiating test on {model_name} runner...")
            print("-" * 50)
            for city in cities:
                print(f"[Requesting]: {city}")
                message_content = types.Content(
                    parts=[types.Part.from_text(text=city)]
                )
                try:
                    events = runner.run(
                        user_id="tester_widget",
                        session_id=f"widget_session_{model_name.lower().replace(' ', '_')}",
                        new_message=message_content,
                    )
                    agent_response = []
                    for event in events:
                        if event.content and event.content.parts:
                            for part in event.content.parts:
                                if part.text:
                                    agent_response.append(part.text)

                    full_response = "".join(agent_response)
                    print(f"[Agent Response]: {full_response.strip()}\n")
                except Exception as e:
                    print(f"Error fetching weather for {city}: {e}\n")

                # Standard sleep to prevent API Rate limits (429)
                time.sleep(10.0)
            print("=" * 50 + "\n")

        print("=== Test Suite Completed Successfully ===")

    test_button.disabled = False

test_button.on_click(run_suite_on_button_click)

# Display widget container
widget_container = widgets.VBox([
    widgets.HTML("<b style='font-size: 14px;'>Programmatic Verification Dashboard:</b><br>"),
    test_button,
    widgets.HTML("<br><b>Execution Output:</b>"),
    test_output
], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='10px 0px'))

ipy_display(widget_container)